In [14]:
from pathlib import Path

In [15]:
gt_directory = Path("video/gt")
log_directory = Path("video/logs")

gt_directory.mkdir(parents=True, exist_ok=True)
log_directory.mkdir(parents=True, exist_ok=True)

In [16]:
def calculate_tiou(seg1, seg2):
    s1, e1 = seg1
    s2, e2 = seg2
    intersection = max(0, min(e1, e2) - max(s1, s2))
    union = (e1 - s1) + (e2 - s2) - intersection
    return intersection / union if union > 0 else 0.0

def load_events(filepath):
    events = {}
    with filepath.open('r') as f:
        for line in f:
            if not line.strip(): 
                continue
            cols = line.strip().split(',')
            
            key = (cols[0], cols[1], cols[2])
            seg = (int(cols[3]), int(cols[4]))
            
            if key not in events:
                events[key] = []
            events[key].append(seg)
    return events

In [17]:
output_file = Path(gt_directory / "evaluation_results.txt")

with output_file.open('w') as out_f:
    for log_file in log_directory.glob("*_interactions_summary.csv"):
        
        video_name = log_file.name.replace("_interactions_summary.csv", "")
        
        gt_file = gt_directory / f"{video_name}_gt.csv"
        
        if not gt_file.exists():
            out_f.write(f"\nSkipping {video_name}: No matching ground truth file found at {gt_file}\n")
            continue
            
        out_f.write(f"\n=== Evaluating Video: {video_name} ===\n")
        
        gt_events = load_events(gt_file)
        log_events = load_events(log_file)
        
        for key in log_events:
            h, o, c = key
            
            if str(h).strip().lower() in ['none', 'null', 'nan', ''] or str(o).strip().lower() in ['none', 'null', 'nan', '']:
                for log_seg in log_events[key]:
                    out_f.write(f"ID({h},{o}) Class:{c} | Log:{log_seg} | tIoU: 0.000 (Null ID Detected)\n")
            elif key in gt_events:
                for log_seg in log_events[key]:
                    best_tiou = 0.0
                    best_gt_seg = None
                    
                    for gt_seg in gt_events[key]:
                        tiou = calculate_tiou(log_seg, gt_seg)
                        if tiou > best_tiou:
                            best_tiou = tiou
                            best_gt_seg = gt_seg
                    
                    out_f.write(f"ID({h},{o}) Class:{c} | Log:{log_seg} matched GT:{best_gt_seg} | tIoU: {best_tiou:.3f}\n")
            else:
                for log_seg in log_events[key]:
                    out_f.write(f"ID({h},{o}) Class:{c} | Log:{log_seg} | tIoU: 0.000 (Not in Ground Truth)\n")

        for key in gt_events:
            if key not in log_events:
                for gt_seg in gt_events[key]:
                    h, o, c = key
                    out_f.write(f"ID({h},{o}) Class:{c} | GT:{gt_seg} | tIoU: 0.000 (Not in Summary)\n")
            else:
                for gt_seg in gt_events[key]:
                    matched = False
                    for log_seg in log_events[key]:
                        if calculate_tiou(log_seg, gt_seg) > 0:
                            matched = True
                            break
                    if not matched:
                        h, o, c = key
                        out_f.write(f"ID({h},{o}) Class:{c} | GT:{gt_seg} | tIoU: 0.000 (Not in Summary)\n")

print(f"Results written to {output_file}")

Results written to video\gt\evaluation_results.txt
